<a href="https://colab.research.google.com/github/justin92102512-crypto/-/blob/main/B12092202_%E6%9D%8E%E6%B0%B8%E9%9D%96_0702_Colab_LINE_Bot_with_GEMINI_Stateful.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.9/818.9 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 6.3 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051


In [ ]:
import os
from pyngrok import ngrok

In [ ]:
ngrok.kill()

In [ ]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://unminding-fibrotic-paulita.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://unminding-fibrotic-paulita.ngrok-free.dev


True

In [ ]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        response_modalities=["TEXT"],
    )
)

In [ ]:
def stateful_query(payload):
    response = chat.send_message(message=payload)
    return response.text

In [ ]:
result = stateful_query("簡介明新科技大學")
print(result)

明新科技大學（Ming Hsin University of Science and Technology, MHUST）是一所位於台灣新竹縣的私立科技大學。

**主要簡介：**

1.  **創立背景與發展：**
    *   創立於**1966年**，前身為明新工業專科學校。
    *   後歷經改制為明新技術學院，最終於**2002年**升格為明新科技大學。
    *   擁有超過半世紀的辦學歷史，是台灣技職教育體系中具代表性的學校之一。

2.  **地理位置優勢：**
    *   學校緊鄰**新竹科學園區**，這個地利優勢讓學校與周邊的高科技產業保持密切的產學合作關係。
    *   為學生提供了豐富的實習與就業機會，也讓學校課程能更貼近產業需求。

3.  **辦學特色與理念：**
    *   **實務導向：** 學校秉持「實務教學」為核心，強調理論與實作並重，培養學生成為具備即戰力的專業人才。
    *   **產學合作：** 積極與企業界合作，推動建教合作、實習課程、共同研發等，使教學內容符合產業脈動。
    *   **就業導向：** 以提升學生的就業競爭力為目標，畢業生在業界有良好的口碑與就業率。

4.  **學院與系所：**
    *   目前設有：
        *   **工程學院：** 涵蓋電機、電子、機械、資工、土木等傳統與新興工程領域。
        *   **管理學院：** 提供企業管理、資訊管理、行銷與流通管理、財務金融等課程。
        *   **服務產業學院：** 包含旅館管理、餐飲管理、幼兒保育、時尚造型等，因應服務業發展趨勢。
        *   **人文社會學院：** 注重通識教育及人文素養的培養。
    *   提供學士、碩士等多層次學制。

5.  **校園環境與設施：**
    *   校園廣闊，環境優美，擁有現代化的教學大樓、實驗室、實習工廠、圖書館、體育設施及學生宿舍等。
    *   致力於提供學生優質的學習與生活環境。

**總結來說，** 明新科技大學是一所強調與產業接軌、注重實務技能培養的科技大學，尤其受惠於新竹科學園區的地利之便，致力於為高科技及服務產業培育具備專業能力與創新思維的人才。


In [ ]:
result2 = stateful_query("校長是誰？")
print(result2)

明新科技大學的現任校長是 **吳菊 校長**。

吳菊校長於2022年8月1日就任明新科技大學第八任校長。


In [ ]:
from flask import Flask, request, abort

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=['POST'])
def callback():
    # get X-Line-Signature header value
    signature = request.headers['X-Line-Signature']

    # get request body as text
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    # handle webhook body
    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature. Please check your channel access token/channel secret.")
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    text = event.message.text
    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)
        if text.startswith('AI '):
            prompt = text[3:]
            reply_text = stateful_query(prompt)
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=reply_text)]
                )
            )

        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=event.message.text),
                        TextMessage(text=event.message.text)]
                )
            )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [07/Jan/2026 09:22:41] "POST / HTTP/1.1" 200 -


BODY:  {"destination":"U96394896c06e15def781d1b318fc78e0","events":[]}
BODY:  {"destination":"U96394896c06e15def781d1b318fc78e0","events":[{"type":"message","message":{"type":"text","id":"595501381658346335","quoteToken":"TJBmq2UF31GSu-DxFd1HVoELRXOeRuyhPFY4A6VDy7nWjDFjh--DGAneWuZ9sS4hUOG7t_GdMU_vcLYpeC5KJEbOFCIlCANwCmV-xe4yMQvUu0v9dTiz-qsT8SOPGIp3fgUC4uJLUr5ReTucHGJpwA","markAsReadToken":"Ut4e0R4FqNEYW9S0yCHMC0ypVugYkbJGYkebkto_1DbuCy4r0LGyjTsjavwJFEzEjiTbEUJK1jSPnEoHrZH_y_Xx6kfn47gVfQ39kp3mAAnNIbBTqqwIGaVUIXVdu2jcCs2-UKeeOzD5eheutC7a20j2SN6LKrIqnUh6k5kuxa3RofzwUHBidFSXEQeOsSZS-i6CnBHwbwQkMtdRBt_syQ","text":"AI 請問明新科技大學校長是誰"},"webhookEventId":"01KEBW8R2B336NAZ6YZDMTK1JX","deliveryContext":{"isRedelivery":false},"timestamp":1767777787372,"source":{"type":"user","userId":"Ub61cc5701cb6aa22ef67c1ca3c38052f"},"replyToken":"0ec1e7dd59644967b0f4073e535c6212","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [07/Jan/2026 09:23:09] "POST / HTTP/1.1" 200 -


BODY:  {"destination":"U96394896c06e15def781d1b318fc78e0","events":[{"type":"message","message":{"type":"text","id":"595501422225653870","quoteToken":"niCxS48yZq6VcznPvmJD_MoamFXO3HcvZ1OLeOE-Wkjld2MHr7TVpvubiON8qDt1uJQMz4ntkIQjhF4RgrSzF3RiYwMFaDmDgt8AK4TGEhbFi6yW1U9tKLQ7YH5wfcb0yRx4c58NxHKtTEokpdZSIQ","markAsReadToken":"E1t_0Jib86SrvcyJY-Vpg0ExDeoz8jnj6fGFXwnrj3KG9d7Nbxf7sfwh_LJMkL36Kv-Ei8-WeaOKmJ9hYm3fQpKa5DjZvmt1UsLu_z2KcIlK3-vzvU9a2C4piZvJNTaqpYwLYJVqWR3jZh6u3d0mDPyFY8CXWaGUb5cLac5Q1MYxzRitxa1TGm6F-DWCzW_8gGqv9iQXTcMCWdPltttxiw","text":"Ai 幫我介紹明新科技大學"},"webhookEventId":"01KEBW9FBEBWRCK4AFVVNYTQ9V","deliveryContext":{"isRedelivery":false},"timestamp":1767777811497,"source":{"type":"user","userId":"Ub61cc5701cb6aa22ef67c1ca3c38052f"},"replyToken":"daca08ae1fc244b996c0f2429c0218de","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [07/Jan/2026 09:23:32] "POST / HTTP/1.1" 200 -


BODY:  {"destination":"U96394896c06e15def781d1b318fc78e0","events":[{"type":"message","message":{"type":"text","id":"595501473749008742","quoteToken":"DrPmVzbxodU3Y8FiAEaJVPNxwt3NIeP1A5o7La8F8EpaE7Xp_Y5wBdErL8Hlukj-GsvEQolg1i0vz6mxMzfMnsEXDWS87Wjh8SSXNlXl6hv1ENYPIWtCarIJGUiN5Le8Qt78ck3JkWAbAC28658YMw","markAsReadToken":"valYwF8lsDlcOZW2gZMz9JjK-0ukgsLN9748Ia-nArOJvQe0ZY46Z_zwc6XkxLMJOQQR0NH762QkEFsfJeMxhS6rw-5bRsO9zz7yvnS6Sdz4FV4C9V1fSz_CDuLp-La7dwlxltmeoyIE6eB3mmXJai-Hq03yTWZBMcHn98fRhBcv8zX0Y2LC5EdUK38lW9ijlMI9ozDvPaYk1ZdLLBvJiw","text":"簡介明新科技大學"},"webhookEventId":"01KEBWAD17S3FWQ9DHTTTD6594","deliveryContext":{"isRedelivery":false},"timestamp":1767777842070,"source":{"type":"user","userId":"Ub61cc5701cb6aa22ef67c1ca3c38052f"},"replyToken":"7e1520a9077c475ea321cfb2bba56dfc","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [07/Jan/2026 09:24:02] "POST / HTTP/1.1" 200 -
